# NB-12-FUTURES · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-12-FUTURES**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-12-FUTURES.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This checks the instrument identities needed before applying the European cross-commodity example to real futures bars. Delivery periods, units, sessions, and roll constructions must be compatible before a margin is calculated.

- [Volume V, Chapter 1 — Data organization is modeling](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c1)
- [Volume V, Chapter 11 — European power, gas, carbon, and weather](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c11)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-12-FUTURES', require_key=True)
print(setup_status)


# NB-12-FUTURES — Contract identity before a margin
**Question:** what must be checked before combining observed power, gas, and
carbon futures? This companion to NB-12 follows real EWB/Barchart daily bars
from 1 September to 14 October 2026. It creates no synthetic observations.
Prerequisites: price units, delivery periods, and the difference between a daily
label and an information timestamp. We inspect contracts, plot separate prices,
audit sessions and roll mappings, and decide whether a margin is admissible.
The retained observations and repository-local helper are required to rerun.

In [ ]:
NOTEBOOK_ID='NB-12-FUTURES'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

f=copy.deepcopy(inputs['inspection'])
f['close']=np.asarray(f['close']);f['times']=[datetime.fromisoformat(t) for t in f['times']]
indices=[]
for ident in inputs['contract_ids']:
    entry=inputs['requests'][ident]
    result=execute(ident,entry['method'],entry['request'])['transformed_series']
    assert result['instrument_id']==ident and result['row_ids']==f['days']
    indices.append(result['values'])
for s in f['spec']:
    print({k:s[k] for k in ('instrument_id','market','delivery_period','currency','unit','tick','multiplier_rule','contract_multiplier')})
print('Common observed dates:',len(f['days']),'unmatched dates:',f['unmatched'])

## 1. Exact contracts, distinct physical quantities
January 2027 German baseload and TTF both quote EUR/MWh, but a MWh of electricity
is not a MWh of fuel. Their retained multipliers are 744 hours per one-MW lot.
December 2026 EUA quotes EUR/tCO2e with a 1,000-unit multiplier. A currency match
is insufficient: a clean-spark calculation also needs a declared efficiency,
emissions intensity, carbon inventory/carry treatment, and compatible delivery
boundaries. A shared month label alone does not prove identical gas and power
delivery hours. We do not introduce plant parameters or a carbon carry assumption.

The catalog's first/last trade dates summarize observed coverage. In particular,
an October last observation for a January contract is not a verified expiry.
Vendor expiration is null in these selected specifications. This snapshot's
reference records are not point-in-time exchange specifications.

In [ ]:
plt=setup_plot();colors=['#88C0D0','#A3BE8C','#EBCB8B']
fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True)
for j,ax in enumerate(axes):
    ax.plot(f['times'],f['close'][:,j],color=colors[j],marker='.',linewidth=1)
    ax.set(title=f['spec'][j]['instrument_id']+' · '+f['spec'][j]['delivery_period'],ylabel=f['spec'][j]['unit']);ax.grid(alpha=.3)
axes[-1].set_xlabel('Trade date label · not an availability time')
fig.autofmt_xdate();fig.tight_layout();plt.show()

## 2. Common-date traces do not establish synchronous executable prices
There are 32 dates for each contract and no unmatched dates in this bounded
sample. We use the intersection explicitly; no forward filling or invented
holiday observations. Lines connect reported daily values, not continuous quotes.
A rebased close index below compares relative movements in this sample. It is
neither a currency-neutral portfolio nor a roll-adjusted total-return index.
Daily closes are retained vendor fields; we have not independently verified
their exchange settlement or executable-quote status.

Fifteen power records have identical open, high, low, and close, despite positive
reported volume. Preserve that feature rather than interpret it as proof of
zero intraday volatility. No selected close lies outside its reported range.
Volume and open interest are contract counts, not comparable energy exposures.

In [ ]:
for a in f['audit']:print(json.dumps(a))
assert all(a['close_outside_high_low']==0 for a in f['audit'])
assert (f['close'][0]!=0).all()
fig,ax=plt.subplots(figsize=(11,4))
for j,s in enumerate(f['spec']):ax.plot(f['times'],indices[j],label=s['instrument_id'],color=colors[j])
ax.set(ylabel='First common close = 100',xlabel='Trade date',title='Relative movements of three fixed contracts');ax.legend();ax.grid(alpha=.3)
fig.autofmt_xdate();fig.tight_layout();plt.show()

## 3. Three clocks and incomplete session evidence
`ts_utc` is trade-date midnight UTC. `retrieved_utc` is collector receipt;
`published_ts_utc` is workbench publication. The latter two can reflect a
backfill, not the earliest time a market participant could know a close.
Workbench publication lags in this sample range from less than one day to about
33 days. Latest-only bars cannot support an auction-time information set.

Effective-dated retained profiles identify TTF as Europe/Amsterdam 01:00–00:00
and EUA as Europe/London 07:00–17:00. The power profile is absent. These are
configured session records, not proof of daily-price publication times. We do
not manufacture a power close time or apply minute-bar session rules to daily
observations. A common trade date does not establish simultaneous prices.

In [ ]:
for j,ident in enumerate(inputs['contract_ids']):
    print(ident, 'profile on first date:', f['sessions'][j][0])
print('First-date lineage clocks:')
for p in f['panels']:
    r=p[f['days'][0]]
    print({k:r[k] for k in ('contract_symbol','trade_date','ts_utc','retrieved_utc','published_ts_utc')})

## 4. A front label changes its deliverable
The retained roll calendar changes both front identities on 30 September.
German power changes September→October while TTF changes October→November.
Thus even the same date and the word “front” do not make a delivery-matched pair.
The expiry-source labels differ and include fallback rules. These are current
snapshot mappings, not evidence that the rule or mapping was known historically.
Calendar rows include non-trading dates and do not establish price availability.
This notebook's plotted contracts remain fixed; no rolled prices are substituted.

In [ ]:
for r in f['transitions']:print(r)
assert f['spec'][0]['delivery_period']==f['spec'][1]['delivery_period']=='2027-01'
assert f['spec'][2]['delivery_period']=='2026-12'
assert f['margin'] is None
print('Clean-spark margin: WITHHELD — delivery/carry and synchronous-price evidence incomplete.')

## Exercise and interpretation
What additional evidence would make a margin meaningful? Specify the plant's
conversion parameters, hourly delivery definitions, carbon inventory/carry,
and a defensible valuation timestamp. Then establish price availability for
each input at that timestamp. Reusing a front series or shifting midnight to
a guessed close cannot supply those facts. NB-12's existing controlled arithmetic
remains the place to study the conversion formula; this observed companion
demonstrates the prerequisites for applying it to market data.

Source contracts: retained EWB descriptors for `bars_1d`, `instruments`,
`sessions`, and `roll_calendar`; market-bars snapshot `20261015T185749Z`.
Observations are vendor-reported, not independently reconciled exchange records.

In [ ]:
for control in inputs['negative_controls']:reject(control['method'],control['request'])
checked=dict(inspection=inputs['inspection'],indices=indices)
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(observed_only=True,fixed_contracts=True,no_imputation=True,margin_withheld=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)